# GateIO — RoNIN baseline + per-fold spread on LOFO

Adds **RoNIN** (Herath et al., 2020), the standard learned inertial-odometry baseline, evaluated leakage-free on the same five leave-one-flight-out (LOFO) folds as GateIO, and produces **per-fold spread** (mean +/- std across folds) and an error-bar figure for all models.

RoNIN is a 1D ResNet-18 that regresses velocity from the IMU window, with no last-known-velocity input, no outage flag, no yaw-rate gate, and no residual head. For a controlled comparison we give it the same 10 IMU channels (accel, gyro, quaternion) GateIO's encoder sees; the quaternion is what lets it predict navigation-frame velocity (the original RoNIN uses a heading-agnostic frame instead). The model is selected by **validation drift**, the same criterion GateIO uses, so the comparison is apples to apples.

**Prerequisites:** the five fold files (`fold0.npz .. fold4.npz`) and your GateIO/LSTM checkpoints (`gateio_v2_best.pt`, `lstm_v2_best.pt`) in `CKPT_DIR/foldK/`. Set the two paths in the setup cell.

## Setup (run once)

In [ ]:
import os, sys

REPO = "/content/gateio"
FOLDS_DIR = "/content/drive/MyDrive/gateio/folds"              # fold0.npz .. fold4.npz  <-- set this
CKPT_DIR  = "/content/drive/MyDrive/data/checkpoints_lofo"     # foldK/gateio_v2_best.pt etc.  <-- set this

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

if not os.path.isdir(REPO):
    !git clone https://github.com/nandini1612/gateio.git {REPO}
%cd {REPO}
!git pull
!pip -q install -r requirements.txt

assert all(os.path.exists(f"{FOLDS_DIR}/fold{k}.npz") for k in range(5)), \
    f"Missing fold npz files in {FOLDS_DIR}"
print("Folds and repo ready.")

## RoNIN model (1D ResNet-18 velocity regressor)

In [ ]:
import torch, torch.nn as nn, numpy as np

class BasicBlock1d(nn.Module):
    def __init__(self, inp, out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(inp, out, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm1d(out)
        self.conv2 = nn.Conv1d(out, out, 3, 1, 1, bias=False);       self.bn2 = nn.BatchNorm1d(out)
        self.relu = nn.ReLU(inplace=True)
        self.down = None
        if stride != 1 or inp != out:
            self.down = nn.Sequential(nn.Conv1d(inp, out, 1, stride, bias=False), nn.BatchNorm1d(out))
    def forward(self, x):
        idn = x
        o = self.relu(self.bn1(self.conv1(x)))
        o = self.bn2(self.conv2(o))
        if self.down is not None: idn = self.down(x)
        return self.relu(o + idn)

class RoNINResNet18(nn.Module):
    """1D ResNet-18 mapping an IMU window (in_ch x WIN_LEN) to a velocity vector."""
    def __init__(self, in_ch=10, out_dim=3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, 64, 7, 2, 3, bias=False), nn.BatchNorm1d(64),
                                  nn.ReLU(inplace=True), nn.MaxPool1d(3, 2, 1))
        self.layer1 = self._make(64, 64, 2, 1)
        self.layer2 = self._make(64, 128, 2, 2)
        self.layer3 = self._make(128, 256, 2, 2)
        self.layer4 = self._make(256, 512, 2, 2)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.5), nn.Linear(512, out_dim))
    def _make(self, inp, out, n, stride):
        layers = [BasicBlock1d(inp, out, stride)] + [BasicBlock1d(out, out) for _ in range(n - 1)]
        return nn.Sequential(*layers)
    def forward(self, x):                       # x: (B, in_ch, WIN_LEN)
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.head(x)

# shared protocol constants / helpers from the repo, so RoNIN matches evaluate_lofo exactly
sys.path.insert(0, "/content/gateio")
from models.gateio import SEQ_LEN, DT, N_IMU_CHAN          # noqa
from eval.evaluate_lofo import classify, OE_LEN            # noqa
N_IN = N_IMU_CHAN                                          # 10 = accel, gyro, quaternion
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def integrate(v_xy, s):
    S = len(v_xy); pos = np.zeros((S, 2), np.float32)
    for t in range(s + 1, S):
        pos[t] = pos[t - 1] + v_xy[t] * DT
    return pos

print("device:", device, "| RoNIN params:",
      sum(p.numel() for p in RoNINResNet18(N_IN, 3).parameters()))

## 1 — Train RoNIN on every LOFO fold (selected by validation drift)

Per-window velocity regression on the training flights, Huber loss, and the checkpoint is chosen by **validation endpoint drift** (the same criterion GateIO uses) rather than raw velocity loss, so the comparison is fair. Writes `ronin_vd_best.pt` per fold. Skips a fold if that checkpoint already exists.

In [ ]:
import time
from torch.utils.data import TensorDataset, DataLoader

def norm_imu(X, Xmed, Xiq):
    Xi = X[:, :, :N_IN].astype(np.float32)
    denom = np.where(Xiq[:N_IN] < 1e-6, 1.0, Xiq[:N_IN]).astype(np.float32)
    return (Xi - Xmed[:N_IN]) / denom

@torch.no_grad()
def val_drift(model, Xv, Yv, vi, Xmed, Xiq, Ymed, Yiqr, cap=150):
    model.eval()
    os_ = SEQ_LEN // 3; oe_ = min(os_ + OE_LEN, SEQ_LEN)
    denom = np.where(Xiq[:N_IN] < 1e-6, 1.0, Xiq[:N_IN]).astype(np.float32)
    vi = np.asarray(vi)
    if len(vi) > cap:
        vi = vi[np.linspace(0, len(vi) - 1, cap).astype(int)]
    drs = []
    for st in vi:
        st = int(st); xrs = Xv[st:st + SEQ_LEN]; yrs = Yv[st:st + SEQ_LEN]
        if len(xrs) < SEQ_LEN:
            continue
        xin = ((xrs[:, :, :N_IN] - Xmed[:N_IN]) / denom).transpose(0, 2, 1)
        pv = model(torch.from_numpy(xin[os_:oe_]).to(device)).cpu().numpy()
        v = pv * Yiqr + Ymed
        vf = np.zeros((SEQ_LEN, 2), np.float32); vf[os_:oe_] = v[:, :2]
        pp = integrate(vf, os_); pt = integrate(yrs[:, :2].astype(np.float32), os_)
        drs.append(np.linalg.norm(pp[oe_ - 1] - pt[oe_ - 1]))
    return float(np.mean(drs))

EPOCHS = 40
huber = nn.SmoothL1Loss(reduction="none")

for k in range(5):
    ckpt = f"{CKPT_DIR}/fold{k}/ronin_vd_best.pt"
    if os.path.exists(ckpt):
        print(f"fold {k}: ronin_vd_best.pt exists - skipping"); continue
    d = np.load(f"{FOLDS_DIR}/fold{k}.npz")
    Xmed = d["X_median"].astype(np.float32); Xiq = d["X_iqr"].astype(np.float32)
    Ymed = d["Y_median"].astype(np.float32); Yiqr = d["Y_iqr"].astype(np.float32)
    Xtr = norm_imu(d["X_train"], Xmed, Xiq).transpose(0, 2, 1)
    Ytr = (d["Y_train"].astype(np.float32) - Ymed) / Yiqr
    Wtr = d["W_train"].astype(np.float32) if "W_train" in d.files else np.ones(len(Xtr), np.float32)
    Xv = d["X_val"].astype(np.float32); Yv = d["Y_val"].astype(np.float32); vvi = d["val_valid_idx"]

    dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(Ytr), torch.from_numpy(Wtr)),
                    batch_size=256, shuffle=True, drop_last=True)
    model = RoNINResNet18(N_IN, 3).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, steps_per_epoch=len(dl), epochs=EPOCHS)
    best = 1e9; os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    print(f"\n===== training RoNIN, fold {k} (test={str(d['split_test'])}) =====  train windows {len(Xtr)}")
    for ep in range(1, EPOCHS + 1):
        model.train()
        for xb, yb, wb in dl:
            xb = xb.to(device); yb = yb.to(device); wb = wb.to(device)
            opt.zero_grad()
            loss = (huber(model(xb), yb).mean(1) * wb).mean()
            loss.backward(); opt.step(); sched.step()
        vd = val_drift(model, Xv, Yv, vvi, Xmed, Xiq, Ymed, Yiqr)
        if vd < best:
            best = vd; torch.save({"state_dict": model.state_dict(), "N_IN": N_IN}, ckpt)
        if ep == 1 or ep % 5 == 0:
            print(f"  ep {ep:>3}  val_drift {vd:6.2f} m  best {best:6.2f} m")
    print(f"  saved -> {ckpt}  (best val drift {best:.2f} m)")

## 2 — Evaluate RoNIN on LOFO (same protocol) with per-fold means

In [ ]:
import csv

pooled = []; csv_rows = []; perfold = {}
for k in range(5):
    d = np.load(f"{FOLDS_DIR}/fold{k}.npz")
    Xt = d["X_test"].astype(np.float32); Yt = d["Y_test"].astype(np.float32); ti = d["test_valid_idx"]
    Xmed = d["X_median"].astype(np.float32); Xiq = d["X_iqr"].astype(np.float32)
    Ymed = d["Y_median"].astype(np.float32); Yiqr = d["Y_iqr"].astype(np.float32)
    flight = str(d["split_test"])
    ck = torch.load(f"{CKPT_DIR}/fold{k}/ronin_vd_best.pt", map_location=device)
    model = RoNINResNet18(ck.get("N_IN", N_IMU_CHAN), 3).to(device)
    model.load_state_dict(ck["state_dict"]); model.eval()
    os_ = SEQ_LEN // 3; oe_ = min(os_ + OE_LEN, SEQ_LEN)
    denom = np.where(Xiq[:N_IMU_CHAN] < 1e-6, 1.0, Xiq[:N_IMU_CHAN]).astype(np.float32)
    fold_d = []
    for si in range(len(ti)):
        st = int(ti[si]); xrs = Xt[st:st + SEQ_LEN]; yrs = Yt[st:st + SEQ_LEN]
        if len(xrs) < SEQ_LEN:
            continue
        xin = ((xrs[:, :, :N_IMU_CHAN] - Xmed[:N_IMU_CHAN]) / denom).transpose(0, 2, 1)
        with torch.no_grad():
            pv = model(torch.from_numpy(xin[os_:oe_]).to(device)).cpu().numpy()
        v_ms = pv * Yiqr + Ymed
        v_full = np.zeros((SEQ_LEN, 2), np.float32); v_full[os_:oe_] = v_ms[:, :2]
        pp = integrate(v_full, os_); pt = integrate(yrs[:, :2].astype(np.float32), os_)
        drift = float(np.linalg.norm(pp[oe_ - 1] - pt[oe_ - 1]))
        grp = classify(xrs, os_, oe_)
        pooled.append((grp, drift)); csv_rows.append((k, flight, grp, drift)); fold_d.append(drift)
    perfold[k] = (flight, float(np.mean(fold_d)))
    print(f"fold {k} ({flight}): {len(fold_d)} sequences, mean {np.mean(fold_d):.1f} m")

a = np.array([dr for _, dr in pooled], float); g = np.array([grp for grp, _ in pooled])
def line(name, mask):
    b = a[mask]
    if len(b):
        print(f"  {name:<10}{b.mean():>9.2f} m   %<5m {100*np.mean(b < 5):>4.0f}   N {len(b)}")
print("\nRoNIN LOFO pooled:")
line("STRAIGHT", g == "STRAIGHT"); line("TURN", g == "TURN"); line("all", np.ones(len(a), bool))
q25, med, q75 = np.percentile(a, [25, 50, 75])
print(f"  spread     median {med:.2f} m   IQR [{q25:.2f}, {q75:.2f}] m")
pf_vals = np.array([perfold[k][1] for k in sorted(perfold)])
print(f"  per-fold mean +/- std over 5 folds: {pf_vals.mean():.1f} +/- {pf_vals.std(ddof=1):.1f} m")

with open("/content/gateio/results/ronin_lofo.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["fold", "test_flight", "group", "ronin_m"]); w.writerows(csv_rows)
print("saved results/ronin_lofo.csv")

## 3 — Refresh the GateIO/LSTM/EKF/Const-v table with the tuned EKF + per-fold spread

Re-runs `evaluate_lofo.py` with EKF tuning **on** (now capped so it finishes on the large validation flight) and writes both `results/lofo_summary.csv` (per sequence) and `results/lofo_summary_perfold.csv` (per-fold means). This is what makes the paper's "tuned EKF" label correct.

In [ ]:
!python -u eval/evaluate_lofo.py \
    --folds-dir {FOLDS_DIR} \
    --ckpt-dir {CKPT_DIR} \
    --lstm \
    --out results/lofo_summary.csv

## 4 — Combined per-fold table + error-bar figure (all five models)

Reads `lofo_summary.csv` (GateIO-TCN, GateIO-LSTM, EKF, Const-v) and `ronin_lofo.csv` (RoNIN), prints per-fold means and mean +/- std across folds, and saves `results/figures/fig_perfold.png`.

In [ ]:
import numpy as np, csv
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

def read(path):
    with open(path) as f:
        return list(csv.DictReader(f))
S = read("/content/gateio/results/lofo_summary.csv")
R = read("/content/gateio/results/ronin_lofo.csv")
folds = sorted(set(int(x["fold"]) for x in S))

def perfold(rows, key):
    out = {}
    for k in folds:
        vals = [float(x[key]) for x in rows
                if int(x["fold"]) == k and x.get(key, "") not in ("", "nan")]
        out[k] = float(np.mean(vals)) if vals else np.nan
    return out

models = {"GateIO-LSTM": perfold(S, "lstm_m"), "GateIO-TCN": perfold(S, "gateio_m"),
          "RoNIN": perfold(R, "ronin_m"), "EKF": perfold(S, "ekf_m"), "Const-v": perfold(S, "constv_m")}
colors = ["#0072B2", "#56B4E9", "#009E73", "#D55E00", "#8aa0b2"]

print("Per-fold mean drift (all sequences):")
print("  model        " + "".join(f"  f{k}" for k in folds) + "     mean +/- std")
for m, pf in models.items():
    vals = np.array([pf[k] for k in folds])
    print(f"  {m:<12}" + "".join(f"{pf[k]:6.1f}" for k in folds)
          + f"    {np.nanmean(vals):5.1f} +/- {np.nanstd(vals, ddof=1):4.1f} m")

means = [np.nanmean([pf[k] for k in folds]) for pf in models.values()]
stds  = [np.nanstd([pf[k] for k in folds], ddof=1) for pf in models.values()]
fig, ax = plt.subplots(figsize=(7.8, 4.7), dpi=150)
x = np.arange(len(models))
ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor="white")
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(list(models.keys()), rotation=12)
ax.set_ylabel("Mean endpoint drift (m)")
ax.set_title("LOFO drift across 5 folds (bar = mean over folds, error bar = std)")
for xi, m, s in zip(x, means, stds):
    ax.text(xi, m + s, f"{m:.1f}", ha="center", va="bottom", fontsize=8.5, color="#12324a")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.grid(axis="y", alpha=0.25, which="both")
fig.tight_layout()
fig.savefig("/content/gateio/results/figures/fig_perfold.png", bbox_inches="tight", facecolor="white")
print("wrote results/figures/fig_perfold.png")
from IPython.display import Image
Image("/content/gateio/results/figures/fig_perfold.png", width=640)

## After running

Files to download into your local repo:
- `results/ronin_lofo.csv`, `results/lofo_summary.csv`, `results/lofo_summary_perfold.csv`
- `results/figures/fig_perfold.png`

Send me the **RoNIN LOFO pooled** block, the **tuned-EKF** pooled row (from cell 3), and the **per-fold table** (cell 4). I will update the paper: correct the EKF label to tuned, add the per-fold mean +/- std, and slot in the error-bar figure.

Then commit yourself:

```bash
git add notebooks/06_ronin_baseline.ipynb eval/ekf_baseline.py eval/evaluate_lofo.py \
        results/ronin_lofo.csv results/lofo_summary.csv results/lofo_summary_perfold.csv \
        results/figures/fig_perfold.png
git commit -m "RoNIN selected by val drift; tuned per-fold EKF; per-fold spread + figure"
git push
```